In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from dtaidistance import dtw
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from statsmodels.tsa.seasonal import seasonal_decompose

from utils.Preprocessing    import PreProcessData
from utils.plotFigures      import linePlot, lineAndScatterPlot, scatterPlot3D

In [2]:
df_raw = pd.read_hdf('data/3200CR001_26-3-2024.h5')

In [3]:
df = df_raw.copy()
df.drop_duplicates(subset='tiempo', inplace=True)
df.loc[:, df.select_dtypes('number').columns] = df.select_dtypes('number').interpolate(method='linear')
df.set_index('tiempo', inplace=True)

In [4]:
df_aggregation = df.select_dtypes('number').apply(lambda variable: variable.resample('h').mean())
df_aggregation.fillna(0, inplace=True)

In [5]:
df_derivative = df_aggregation.apply(lambda variable: variable.diff())
df_derivative.dropna(inplace=True)

In [6]:
df_trend = df_aggregation.apply(lambda variable: seasonal_decompose(variable, model = 'additive', extrapolate_trend='freq').trend)
df_derivative_trend = df_derivative.apply(lambda variable: seasonal_decompose(variable, model = 'additive', extrapolate_trend='freq').trend)

In [7]:
dataframe = df_raw.select_dtypes('number').rolling(window=30, min_periods=1).median()
dataframe['tiempo'] = df_raw['tiempo']
dataframe = dataframe[dataframe['tiempo'].between('2023-01-10 00:00:00', '2023-01-11 00:00:00')]
variable = dataframe.columns[20]
fig = go.Figure()
fig.add_trace(go.Scatter(x=dataframe['tiempo'], 
                         y=dataframe[variable], 
                         mode='lines', 
                         name='normal'))
fig.update_layout(title=f'{variable}')
fig.show()

In [8]:
df_derivative_trend.apply(lambda variable: np.sign(variable))

,corriente(A),velocidad(rpm),nivel_tolva_compensacion(%),Potencia(kW),posicion_poste(%),rendimiento(ton),presion_estanque_aire(bar),Piñón_ext(°C),Piñón_int(°C),Piñón_int(mm/s),...,Lubricación_entrada_inf(psi),Lubricación_entrada_sup(m3/h),Lubricación_entrada_inf(m3/h),Lubricación_depósito(%),Motor_Fase1(A),Motor_Fase2(A),Motor_Fase3(A),Motor_Fase1(V),Motor_Fase2(V),Motor_Fase3(V)
tiempo,,,,,,,,,,,,,,,,,,,,,
2023-01-01 01:00:00,-1.0,-1.0,1.0,-1.0,1.0,-1.0,1.0,-1.0,-1.0,-1.0,...,-1.0,-1.0,1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
2023-01-01 02:00:00,-1.0,-1.0,1.0,-1.0,1.0,-1.0,1.0,-1.0,-1.0,-1.0,...,-1.0,-1.0,1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
2023-01-01 03:00:00,-1.0,-1.0,1.0,-1.0,1.0,-1.0,1.0,-1.0,-1.0,-1.0,...,-1.0,-1.0,1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
2023-01-01 04:00:00,-1.0,-1.0,1.0,-1.0,1.0,-1.0,1.0,-1.0,-1.0,-1.0,...,-1.0,-1.0,1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
2023-01-01 05:00:00,-1.0,-1.0,1.0,-1.0,1.0,-1.0,1.0,-1.0,-1.0,-1.0,...,-1.0,-1.0,1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-31 20:00:00,-1.0,-1.0,1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,...,-1.0,-1.0,1.0,1.0,-1.0,-1.0,-1.0,1.0,1.0,1.0
2023-12-31 21:00:00,-1.0,-1.0,1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,...,-1.0,-1.0,1.0,1.0,-1.0,-1.0,-1.0,1.0,1.0,1.0
2023-12-31 22:00:00,-1.0,-1.0,1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,...,-1.0,-1.0,1.0,1.0,-1.0,-1.0,-1.0,1.0,1.0,1.0


# -------------------------

In [ ]:
variable = df_trend.columns[5]
data = df_derivative_trend
linePlot(data = data, x=data.index, y=variable, color=None)

In [ ]:
scaler = MinMaxScaler()
df_trend_scale = df_trend.copy()
df_trend_scale.loc[:, df_trend.columns] = scaler.fit_transform(df_trend)

scaler = StandardScaler()
df_derivative_trend_scale = df_derivative_trend.copy()
df_derivative_trend_scale .loc[:, df_derivative_trend.columns] = scaler.fit_transform(df_derivative_trend)

In [ ]:
variable = df_trend_scale.columns[5]
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_trend_scale.index, y=df_trend_scale[variable], mode='lines', name='normal'))
fig.add_trace(go.Scatter(x=df_derivative_trend_scale.index, y=df_derivative_trend_scale[variable], mode='lines', name='derivada'))
fig.update_layout(title=f'{variable}')
fig.show()

In [ ]:
preData = PreProcessData()
preData.dataLoading(typeFile='h5')
df_split      = preData.splitTimeSeries()
# dfRaw = preData.crusher1Raw

# scaler  = StandardScaler()
# preData.normalizeData(scaler);

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
def decompose(df, column_name):
    """
    A function that returns the trend, seasonality and residual captured by applying both multiplicative and
    additive model.
    df -> DataFrame
    column_name -> column_name for which trend, seasonality is to be captured
    """
    #result_mul = seasonal_decompose(df[column_name].resample('h').mean(), model='multiplicative') #, extrapolate_trend = 'freq')
    result_add = seasonal_decompose(df[column_name], model = 'additive') #, extrapolate_trend='freq')

    plt.rcParams.update({'figure.figsize': (20, 10)})
    #result_mul.plot()#.suptitle('Multiplicative Decompose', fontsize=30)
    result_add.plot() #.suptitle('Additive Decompose', fontsize=30)
    plt.show()
    
    #return result_mul, result_add
    return result_add

In [ ]:
# variable = df_aggregation.columns[0]
# result_add = decompose(df_aggregation, variable)